In [26]:
import os
from dotenv import load_dotenv
from huggingface_hub import HfApi
from pathlib import Path

PROJECT_DIR = Path.cwd().resolve().parent
load_dotenv(PROJECT_DIR/".env", override=True)

api = HfApi(token=os.environ["HF_TOKEN"])
REPO_ID = os.environ["HF_REPO_ID"]
print(f"Account: {api.whoami()["name"]}")
print

Account: sdoan17


<function print(*args, sep=' ', end='\n', file=None, flush=False)>

### 1. Locate and check models files

In [27]:
MODEL_DIR = PROJECT_DIR / "models"

filenames =  [
    "spam-model.keras",
    "spam-classifier-tokenizer.json",
    "spam-classifier-metadata.json",
]

for filename in filenames:
    path = MODEL_DIR / filename

    if not path.is_file():
        raise FileNotFoundError(f"Missing model file: {path}")

### 2. Create a prublic repo and upload

In [28]:
from scipy.linalg import fiedler
from huggingface_hub import CommitOperationAdd

api.create_repo(
    repo_id = REPO_ID,
    repo_type = "model",
    exist_ok= True,
)

operations = [
    CommitOperationAdd(
        path_in_repo=filename,
        path_or_fileobj=str(MODEL_DIR / filename),
    )
    for filename in filenames
]

commit = api.create_commit(
    repo_id=REPO_ID,
    repo_type="model",
    operations=operations,
    commit_message="Upload spam model, tokenizer, and metadata",
)

MODEL_REVISION = commit.oid
print("Repository:", f"https://huggingface.co/{REPO_ID}")
print("Model revision:", MODEL_REVISION)

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Validating:           |   0%            

Repository: https://huggingface.co/sdoan17/spam-classifier
Model revision: 90982047215d4f661fe8b81ce3810c72b54b608d


In [29]:
uploaded_files = api.list_repo_files(
    repo_id=REPO_ID,
    repo_type="model",
    revision=MODEL_REVISION,
)

missing = set(filenames) - set(uploaded_files)

if missing:
    raise RuntimeError(f"Files missing from the repository: {missing}")

print("All three model files uploaded successfully.")
print(f"View them: https://huggingface.co/{REPO_ID}/tree/{MODEL_REVISION}")

All three model files uploaded successfully.
View them: https://huggingface.co/sdoan17/spam-classifier/tree/90982047215d4f661fe8b81ce3810c72b54b608d
